In [1]:
import pathlib
import os

import pandas as pd
import pyreadr
import altair as alt


In [2]:
correction_dir = pathlib.Path('asap-analysis-data') / 'correction-data'

r_objs = {
    'no_correction_no_smor': pyreadr.read_r(
        correction_dir / 'correction-no-smor-asap-data' / 'Combined_ASAP_Data.Rdata'
    ),
    'correction_no_smor': pyreadr.read_r(
        correction_dir / 'correction-no-smor-asap-data' / 'Combined_ASAP_Data.Rdata'
    ),
    'no_correction_smor': pyreadr.read_r(
        correction_dir / 'no-correction-smor-asap-data' / 'Combined_ASAP_Data.Rdata'
    ),
}

dfs = {k: v['final_snps'] for k, v in r_objs.items()}

In [3]:
dfs['correction_no_smor'].head()

,run,name,Mapped_Reads,unassigned_reads,unmapped_reads,assay_function,assay_gene,assay_name,assay_type,amplicon_number,location_depth,snp_name,snp_position,snp_reference,snp_depth,snp_proportion,snp_call,snp_distribution
0,Individual_XML_processing,TB-BDQPTC2-123_S21_L001,30464.0,2414.0,2432.0,species ID,,NC_000962.3,presence/absence,1.0,2992.0,unknown,778961.0,G,6.0,0.200535,A,"G=2980, A=6, T=6"
1,Individual_XML_processing,TB-BDQPTC2-123_S21_L001,30464.0,2414.0,2432.0,species ID,,NC_000962.3,presence/absence,1.0,2992.0,unknown,778970.0,T,11.0,0.367647,G,"T=2976, C=5, G=11"
2,Individual_XML_processing,TB-BDQPTC2-123_S21_L001,30464.0,2414.0,2432.0,species ID,,NC_000962.3,presence/absence,1.0,2992.0,unknown,778976.0,G,8.0,0.267380,T,"G=2981, T=8, A=3"
3,Individual_XML_processing,TB-BDQPTC2-123_S21_L001,30464.0,2414.0,2432.0,species ID,,NC_000962.3,presence/absence,1.0,2992.0,unknown,778982.0,T,7.0,0.233957,A,"T=2978, G=5, A=7, C=2"
4,Individual_XML_processing,TB-BDQPTC2-123_S21_L001,30464.0,2414.0,2432.0,species ID,,NC_000962.3,presence/absence,1.0,2990.0,unknown,778995.0,C,5.0,0.167224,G,"C=2984, G=5, A=1"


In [4]:
'''
Derive a `non_reference_count` column from the `snp_distribution` column, defined as the number of non-reference
base calls. Needed for error rate calculation.
'''

def snp_distribution_to_non_ref_calls(distribution_string: str):
    snps = distribution_string.split(',')
    if len(snps) == 1:
        return 0
        
    assert len(snps) >= 2
    
    snp_counts = [int(snp.split('=')[1]) for snp in snps]
    assert len(snps) == len(snp_counts)
    snp_counts.sort(reverse=True)
    return sum(snp_counts[1:])


assert snp_distribution_to_non_ref_calls('G=10, T=5') == 5
assert snp_distribution_to_non_ref_calls('G=10, C=5, T=9') == 14
assert snp_distribution_to_non_ref_calls('A=10, G=5, C=25') == 15

for df in dfs.values():
    df['non_reference_count'] = df['snp_distribution'].apply(snp_distribution_to_non_ref_calls)

In [5]:
'''
Calculate per-position error rate as sum(`non_reference_count`) / sum(`location_depth`) where the sums contain
the counts across samples. In other words, group by `snp_position`. 
'''

for df in dfs.values():
    df['error_rate'] = \
        df.groupby('snp_position')['non_reference_count'].transform('sum') / \
        df.groupby('snp_position')['location_depth'].transform('sum')

In [6]:
'''
Keep only one snp position from all samples, as error rate is the same.
'''

for k, df in dfs.items():
    dfs[k] = df.groupby('snp_position', as_index=False).first()

In [7]:
'''
Merge the three correction types on `snp_position` and `name`.
'''

keep_cols = ['snp_position', 'name', 'error_rate']
merged = pd.merge(
    dfs['correction_no_smor'][keep_cols],
    dfs['no_correction_no_smor'][keep_cols],
    on=['snp_position', 'name'],
    how='inner',
    suffixes=['_correction_no_smor', '_no_correction_no_smor']
)
merged = pd.merge(
    merged,
    dfs['no_correction_smor'][keep_cols],
    on=['snp_position', 'name'],
    how='inner',
)
merged.rename({'error_rate': 'error_rate_no_correction_smor'}, axis=1, inplace=True)

In [8]:
'''
Need to discuss why snp positions are so spread...
'''
alt.Chart(merged).mark_bar().encode(
    x=alt.X('snp_position:Q', bin=True),
    y='count()'
)

alt.Chart(...)

In [9]:
'''
Redefine `snp_position` as index in sorted list of positions to solve spread issue for now.
'''
merged.sort_values(by='snp_position', inplace=True)
merged['snp_position'] = merged.index
merged.head()

,snp_position,name,error_rate_correction_no_smor,error_rate_no_correction_no_smor,error_rate_no_correction_smor
0,0,TB_PTC-BDQ-Kaward-25_S104_L001,0.466051,0.466051,0.468167
1,1,TB_PTC-BDQ-Kaward-25_S104_L001,0.000997,0.000997,0.001031
2,2,TB_PTC-BDQ-Kaward-25_S104_L001,0.001102,0.001102,0.001041
3,3,TB_DJNitroPTC1E1_S130_L001,0.000735,0.000735,0.000720
4,4,TB_DJNitroPTC1E1_S130_L001,0.000488,0.000488,0.000465


In [16]:
'''
Visualize
'''

# allow plotting more than 500 rows
alt.data_transformers.enable("vegafusion")

merged_long = merged.melt(
    id_vars='snp_position',
    value_vars=[
        'error_rate_no_correction_no_smor',
        'error_rate_correction_no_smor',
        'error_rate_no_correction_smor'
    ],
    var_name='correction_type',
    value_name='error_rate'
)

chart = alt.Chart(merged_long.sample(frac=0.1)).mark_line().encode(
    x=alt.X('snp_position:Q'),
    y=alt.Y('error_rate:Q').scale(type='log'),
    color=alt.Color('correction_type:N')
)

chart

alt.Chart(...)